<a href="https://colab.research.google.com/github/shreya252005/Flight_delay_prediction/blob/main/Flight_delay_prediction_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns

In [17]:
df = pd.read_csv('/Flight_delay.csv')
df.head()

df = df[['DayOfWeek','Date','DepTime','Airline','Origin','Dest','CarrierDelay']]
print("Missing values per column:\n", df.isnull().sum())

df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df['month'] = df['Date'].dt.month
df['day'] = df['Date'].dt.day
df = df.drop(columns=['Date'])

categories = ['Airline', 'Origin', 'Dest']

df_encoded = pd.get_dummies(df, drop_first=True)
df_encoded['is_delayed_60+'] = np.where(df_encoded['CarrierDelay'] > 60, 1, 0)

Missing values per column:
 DayOfWeek       0
Date            0
DepTime         0
Airline         0
Origin          0
Dest            0
CarrierDelay    0
dtype: int64


In [18]:
# Defining Features (X) and Target (y)
X = df_encoded.drop(columns=['is_delayed_60+', 'CarrierDelay'])
y = df_encoded['is_delayed_60+']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

In [19]:
train_set = pd.concat([X_train, y_train], axis=1)

# Reverse One-Hot Encoding for readability
for category in categories:
    one_hot_columns = [col for col in train_set.columns if col.startswith(f'{category}_')]
    train_set[category] = train_set[one_hot_columns].idxmax(axis=1)
    train_set = train_set.drop(columns=one_hot_columns)
    train_set[category] = train_set[category].str.replace(f'{category}_', '')

# Then to check target distribution & mean delay percentage -
print("\nTarget Distribution:\n", train_set['is_delayed_60+'].value_counts())
print("Overall Severe Delay Rate:", round(train_set['is_delayed_60+'].mean() * 100, 2), "%")

# Group delays by Day of Week
day_pct_delayed = train_set.groupby('DayOfWeek')['is_delayed_60+'].mean().round(3) * 100
print("\nDelays by Day of Week (%):\n", day_pct_delayed)
# most delays on 6th day of the week - Saturday


Target Distribution:
 is_delayed_60+
0    313876
1     25309
Name: count, dtype: int64
Overall Severe Delay Rate: 7.46 %

Delays by Day of Week (%):
 DayOfWeek
1    7.6
2    7.1
3    7.6
4    6.9
5    6.8
6    9.0
7    7.9
Name: is_delayed_60+, dtype: float64


In [20]:
# XGBoost Classifier used to initialize and train.
xgb_model = xgb.XGBClassifier(random_state=0, eval_metric='logloss')
xgb_model.fit(X_train, y_train)
print("\nModel training completed successfully!")


Model training completed successfully!


In [23]:
# asking user for flight inputs and converting it into a flight delay prediction system
def predict_flight_delay(model, training_columns):

    day_of_week = int(input("Enter Day of Week (1 = Monday, 7 = Sunday): "))
    month = int(input("Enter Month (1-12): "))
    day = int(input("Enter Day of Month (1-31): "))
    dep_time = int(input("Enter Departure Time (e.g., 1829 for 6:29 PM): "))

    airline = input("Enter Airline Name (e.g., Southwest Airlines Co.): ")
    origin = input("Enter Origin Airport Code (e.g., IND): ")
    dest = input("Enter Destination Airport Code (e.g., BWI): ")

    input_data = pd.DataFrame(0, index=[0], columns=training_columns)
    if 'DayOfWeek' in input_data.columns:
        input_data['DayOfWeek'] = day_of_week
    if 'month' in input_data.columns:
        input_data['month'] = month
    if 'day' in input_data.columns:
        input_data['day'] = day
    if 'DepTime' in input_data.columns:
        input_data['DepTime'] = dep_time

    airline_col = f"Airline_{airline}"
    origin_col = f"Origin_{origin}"
    dest_col = f"Dest_{dest}"

    if airline_col in input_data.columns:
        input_data[airline_col] = 1
    if origin_col in input_data.columns:
        input_data[origin_col] = 1
    if dest_col in input_data.columns:
        input_data[dest_col] = 1

    # To predict the probability of a severe delay (>60 mins)
    delay_probability = model.predict_proba(input_data)[0][1] * 100

    # results
    print("\n" + "="*40)
    print(f"Calculated Delay Probability: {delay_probability:.1f}%")

    if delay_probability > 15:
        print("VERDICT: YES, highly likely to face a severe delay (>60 mins).")
    else:
        print("VERDICT: NO severe delay expected.")
    print("="*40 + "\n")

predict_flight_delay(xgb_model, X_train.columns)

Enter Day of Week (1 = Monday, 7 = Sunday): 5
Enter Month (1-12): 5
Enter Day of Month (1-31): 22
Enter Departure Time (e.g., 1829 for 6:29 PM): 1530
Enter Airline Name (e.g., Southwest Airlines Co.): Southwest Airlines Co.
Enter Origin Airport Code (e.g., IND): JFK
Enter Destination Airport Code (e.g., BWI): BWI

Calculated Delay Probability: 2.1%
VERDICT: NO severe delay expected. Smooth skies ahead!

